In [ ]:
import json
from google.colab import files
files.upload();
!mkdir /root/.kaggle/
!mv kaggle.json /root/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
!kaggle config set -n path -v {/content}

Saving kaggle.json to kaggle.json
mkdir: cannot create directory ‘/root/.kaggle/’: File exists
- path is now set to: {/content}


In [ ]:
!cd /content
!kaggle competitions download -c home-credit-credit-risk-model-stability

 99% 3.13G/3.14G [00:53<00:00, 257MB/s]
100% 3.14G/3.14G [00:53<00:00, 63.1MB/s]


In [ ]:
!ls -lah
!mv '{'/'content}'/competitions/home-credit-credit-risk-model-stability/home-credit-credit-risk-model-stability.zip .

total 20K
drwxr-xr-x 1 root root 4.0K Nov 17 21:40  .
drwxr-xr-x 1 root root 4.0K Nov 17 21:24  ..
drwxr-xr-x 3 root root 4.0K Nov 17 21:40 '{'
drwxr-xr-x 4 root root 4.0K Nov 12 14:30  .config
drwxr-xr-x 1 root root 4.0K Nov 12 14:30  sample_data


In [ ]:
!pwd
!ls -lah
!unzip home-credit-credit-risk-model-stability.zip

/content
total 3.2G
drwxr-xr-x 1 root root 4.0K Nov 17 21:41  .
drwxr-xr-x 1 root root 4.0K Nov 17 21:24  ..
drwxr-xr-x 3 root root 4.0K Nov 17 21:40 '{'
drwxr-xr-x 4 root root 4.0K Nov 12 14:30  .config
-rw-r--r-- 1 root root 3.2G Mar 11  2024  home-credit-credit-risk-model-stability.zip
drwxr-xr-x 1 root root 4.0K Nov 12 14:30  sample_data
Archive:  home-credit-credit-risk-model-stability.zip
  inflating: csv_files/test/test_applprev_1_0.csv  
  inflating: csv_files/test/test_applprev_1_1.csv  
  inflating: csv_files/test/test_applprev_1_2.csv  
  inflating: csv_files/test/test_applprev_2.csv  
  inflating: csv_files/test/test_base.csv  
  inflating: csv_files/test/test_credit_bureau_a_1_0.csv  
  inflating: csv_files/test/test_credit_bureau_a_1_1.csv  
  inflating: csv_files/test/test_credit_bureau_a_1_2.csv  
  inflating: csv_files/test/test_credit_bureau_a_1_3.csv  
  inflating: csv_files/test/test_credit_bureau_a_1_4.csv  
  inflating: csv_files/test/test_credit_bureau_a_2_0.csv 

In [8]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [9]:
!cp -r /content/drive/MyDrive/Kaggle/parquet_files/* /content/parquet_files

In [7]:
!ls /content/parquet_files/train/aggregated/

aggr_applprev_1_0.parquet	  aggr_other_1.parquet
aggr_credit_bureau_a_1_0.parquet  aggr_person_1.parquet
aggr_credit_bureau_b_1.parquet	  aggr_tax_registry_a_1.parquet
aggr_debitcard_1.parquet	  aggr_tax_registry_b_1.parquet
aggr_deposit_1.parquet		  aggr_tax_registry_c_1.parquet


In [40]:
import polars as pl
import os

BASE_DIR = "/content/parquet_files/train"
AGGR_DIR = "/content/parquet_files/train/aggregated"
os.makedirs(AGGR_DIR, exist_ok=True)

AGGR_FILE = os.path.join(AGGR_DIR, "merged.parquet")

def set_table_dtypes(lf: pl.LazyFrame) -> pl.LazyFrame:
    lf_columns = lf.collect_schema().names()
    cast_exprs = []

    for col in lf_columns:
        if col in ["case_id", "WEEK_NUM", "num_group1", "num_group2"]:
            cast_exprs.append(pl.col(col).cast(pl.Int64))
        elif col in ["date_decision"]:
            cast_exprs.append(pl.col(col).cast(pl.Date))
        elif col[-1] in ("P", "A"):
            cast_exprs.append(pl.col(col).cast(pl.Float64))
        elif col[-1] in ("M",):
            cast_exprs.append(pl.col(col).cast(pl.String))
        elif col[-1] in ("D",):
            cast_exprs.append(pl.col(col).cast(pl.Date))
    return lf.with_columns(cast_exprs)

def filter_cols(lf: pl.LazyFrame) -> pl.LazyFrame:
    protected = {"target", "case_id", "WEEK_NUM"}

    # get string columns from schema
    schema = lf.collect_schema()
    str_cols = [
        name
        for name, dtype in zip(schema.names(), schema.dtypes())
        if dtype == pl.Utf8 and name not in protected
    ]

    if not str_cols:
        return lf

    # compute n_unique for string columns
    nunique_df = lf.select(
        [pl.col(c).n_unique().alias(c) for c in str_cols]
    ).collect()

    # drop string cols that have more than 200 unqiue values (phones, IDs etc.)
    drop_cols = []
    for c in str_cols:
        freq = nunique_df[0, c]
        if freq == 1 or freq > 200:
            drop_cols.append(c)

    # return lazy frame with those columns dropped
    print(f"Dropped {len(drop_cols)} columns")
    return lf.drop(drop_cols)

def build_agg_exprs(tbl: pl.LazyFrame, filename: str) -> list:
    """
    Build aggregation expressions for depth>0 table
    """
    cols = tbl.collect_schema().names()

    cols_A = [c for c in cols if c.endswith("A")]
    cols_P = [c for c in cols if c.endswith("P")]
    cols_D = [c for c in cols if c.endswith("D")]
    cols_M = [c for c in cols if c.endswith("M")]
    cols_TL = [c for c in cols if c.endswith(("T", "L"))]

    agg_exprs = [pl.len().alias(f"{filename}_row_count")]

    # Amount-like
    agg_exprs += [
        pl.col(c).sum().alias(f"{c}_sum") for c in cols_A
    ] + [
        pl.col(c).mean().alias(f"{c}_mean") for c in cols_A
    ] + [
        pl.col(c).max().alias(f"{c}_max") for c in cols_A
    ]

    # DPD-like
    agg_exprs += [
        pl.col(c).max().alias(f"{c}_max") for c in cols_P
    ] + [
        pl.col(c).mean().alias(f"{c}_mean") for c in cols_P
    ]

    # Date-like (assuming numeric or Date)
    agg_exprs += [
        pl.col(c).min().alias(f"{c}_min") for c in cols_D
    ] + [
        pl.col(c).max().alias(f"{c}_max") for c in cols_D
    ]

    # Masked categories
    agg_exprs += [
    #    pl.col(c).n_unique().alias(f"{c}_nunique") for c in cols_M
    #Should be unique but this is memory heavy op, for now max
        pl.col(c).max().alias(f"{c}_max") for c in cols_M
    ]

    # Other numeric transforms
    agg_exprs += [
        pl.col(c).mean().alias(f"{c}_mean") for c in cols_TL
    ] + [
        pl.col(c).max().alias(f"{c}_max") for c in cols_TL
    ]

    return agg_exprs

def aggregate_group(group: list):
  """
    Scan parquets of same logical table, aggregate by case_id, write to disk
    Returns aggregated file path
  """

  #read raw files
  raw_paths = [os.path.join(BASE_DIR, f"train_{p}") for p in group]
  tbl = pl.scan_parquet(raw_paths)

  #filter some columns
  tbl = filter_cols(tbl)
  tbl = set_table_dtypes(tbl)

  #aggregate raw files
  filename = os.path.splitext(group[0])[0] #e.g. applprev_1_0
  agg_exprs = build_agg_exprs(tbl,filename)
  features = (
      tbl
      .group_by("case_id")
      .agg(agg_exprs)
  )

  #write aggr file
  agg_path = os.path.join(AGGR_DIR, f"aggr_{filename}.parquet")
  features.sink_parquet(agg_path, mkdir=True)
  print(f"Aggregated {group} -> {agg_path}")

  return agg_path
  #return features
  #features.head().collect()


In [41]:
#
#
# Aggregate every table, remove stupid columns,
#
#

BASE_F = "base.parquet"
DEPTH0_F = [
    #["base.parquet"],
    ["static_0_0.parquet", "static_0_1.parquet"],
    ["static_cb_0.parquet"]
]
DEPTH1_F = [
    [
        "applprev_1_0.parquet",
        "applprev_1_1.parquet",
    ],
    ["other_1.parquet"],
    ["deposit_1.parquet"],
    ["person_1.parquet"],
    ["debitcard_1.parquet"],
    ["tax_registry_a_1.parquet"],
    ["tax_registry_b_1.parquet"],
    ["tax_registry_c_1.parquet"],
    [
        "credit_bureau_a_1_0.parquet",
        "credit_bureau_a_1_1.parquet",
        "credit_bureau_a_1_2.parquet",
        "credit_bureau_a_1_3.parquet",
    ],
    [
        "credit_bureau_b_1.parquet"
    ]
]

#clean aggregated dir
!rm -r /content/parquet_files/train/aggregated/*

#Build base + depth-0
lf = pl.scan_parquet(os.path.join(BASE_DIR, f"train_{BASE_F}"))

for group in DEPTH0_F:
    paths = [os.path.join(BASE_DIR, f"train_{p}") for p in group]
    child = pl.scan_parquet(paths)
    lf = lf.join(child, on="case_id", how="left")
    print(f"Joined depth0 group {group}")

#Aggregate all depth-1 tables to disk
agg_paths = []
for group in DEPTH1_F:
  agg_path = aggregate_group(group)
  agg_paths.append(agg_path)


Joined depth0 group ['static_0_0.parquet', 'static_0_1.parquet']
Joined depth0 group ['static_cb_0.parquet']
Dropped 9 columns
Aggregated ['applprev_1_0.parquet', 'applprev_1_1.parquet'] -> /content/parquet_files/train/aggregated/aggr_applprev_1_0.parquet
Aggregated ['other_1.parquet'] -> /content/parquet_files/train/aggregated/aggr_other_1.parquet
Dropped 2 columns
Aggregated ['deposit_1.parquet'] -> /content/parquet_files/train/aggregated/aggr_deposit_1.parquet
Dropped 9 columns
Aggregated ['person_1.parquet'] -> /content/parquet_files/train/aggregated/aggr_person_1.parquet
Dropped 1 columns
Aggregated ['debitcard_1.parquet'] -> /content/parquet_files/train/aggregated/aggr_debitcard_1.parquet
Dropped 2 columns
Aggregated ['tax_registry_a_1.parquet'] -> /content/parquet_files/train/aggregated/aggr_tax_registry_a_1.parquet
Dropped 2 columns
Aggregated ['tax_registry_b_1.parquet'] -> /content/parquet_files/train/aggregated/aggr_tax_registry_b_1.parquet
Dropped 2 columns
Aggregated ['tax

In [42]:
#
#
# Create batches of flattened rows as parquet files
#
#

n_rows = lf.select(pl.len()).collect()[0, 0]
batch_size = 200_000

for offset in range(0, n_rows, batch_size):
    # get base + depth0 for this batch
    base_batch = lf.slice(offset, batch_size).collect()
    id_df = pl.DataFrame({"case_id": base_batch["case_id"]})

    # join in features from agg files for this slice
    for p in agg_paths:
        feats = (
          pl.scan_parquet(p)
          .join(id_df.lazy(), on="case_id", how="inner")
          .collect()
        )
        base_batch = base_batch.join(feats, on="case_id", how="left")

    out_path = os.path.join(AGGR_DIR, f"batch_{offset:09d}.parquet")
    base_batch.write_parquet(out_path)
    print(f"Wrote batch {offset} to :", out_path)

Wrote batch 0 to : /content/parquet_files/train/aggregated/batch_000000000.parquet
Wrote batch 200000 to : /content/parquet_files/train/aggregated/batch_000200000.parquet
Wrote batch 400000 to : /content/parquet_files/train/aggregated/batch_000400000.parquet
Wrote batch 600000 to : /content/parquet_files/train/aggregated/batch_000600000.parquet
Wrote batch 800000 to : /content/parquet_files/train/aggregated/batch_000800000.parquet
Wrote batch 1000000 to : /content/parquet_files/train/aggregated/batch_001000000.parquet
Wrote batch 1200000 to : /content/parquet_files/train/aggregated/batch_001200000.parquet
Wrote batch 1400000 to : /content/parquet_files/train/aggregated/batch_001400000.parquet


In [44]:
#
#
# Split val, test
#
#

import numpy as np
from glob import glob

paths = sorted(glob(os.path.join(AGGR_DIR, "batch_*.parquet")))

rng = np.random.default_rng(42)
rng.shuffle(paths)

n_batches = len(paths)
n_val = max(1, int(0.1 * n_batches))
n_test = max(1, int(0.1 * n_batches))

val_paths  = paths[:n_val]
test_paths = paths[n_val:n_val + n_test]
train_paths = paths[n_val + n_test:]

print(f"train batches: {len(train_paths)}, val: {len(val_paths)}, test: {len(test_paths)}")

def load_batches(paths):
    dfs = [pl.read_parquet(p) for p in paths]
    return pl.concat(dfs, how="vertical")

def df_to_xy(df: pl.DataFrame):
    df_num = df.with_columns([
        pl.col(pl.Date, pl.Datetime).cast(pl.Int64),
        pl.col(pl.Boolean).cast(pl.Int8),
    ])
    X = df_num.select(
        pl.all()
        .exclude(["target", "case_id"])
        .exclude(pl.Utf8)
    ).to_numpy()
    y = df_num["target"].to_numpy()
    return X, y

val_df  = load_batches(val_paths)
test_df = load_batches(test_paths)

X_val,  y_val  = df_to_xy(val_df)
X_test, y_test = df_to_xy(test_df)

train batches: 6, val: 1, test: 1


In [47]:
#
#
#
# Train by feeding in batches
#
#

import lightgbm as lgb

params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "learning_rate": 0.05,
    "num_leaves": 64,
    "feature_pre_filter": False,  # helpful when doing multiple calls
    "verbosity":-1,
}
dvalid = lgb.Dataset(X_val, label=y_val, free_raw_data=False)
model = None
best_score = float("inf")
no_improve = 0
patience_batches = 3

for i, path in enumerate(train_paths):
    batch = pl.read_parquet(path)

    # convert dates/bools to numeric
    batch_num = batch.with_columns([
        pl.col(pl.Date, pl.Datetime).cast(pl.Int64),  # days since epoch
        pl.col(pl.Boolean).cast(pl.Int8),             # bool to 1/0
    ])

    # keep only numeric columns, drop target & id
    X_train = batch_num.select(
        pl.all()
        .exclude(["target", "case_id"])
        .exclude(pl.Utf8)        # drop string cols for now
    ).to_numpy()

    y_train = batch["target"].to_numpy()

    dtrain = lgb.Dataset(X_train, label=y_train, free_raw_data=False)

    if model is None:
          # first batch: start model
          model = lgb.train(
              params,
              dtrain,
              num_boost_round=100,
              valid_sets=[dvalid],
              valid_names=["valid"],
              keep_training_booster=True
          )
    else:
        # later batches: continue training
        model = lgb.train(
            params,
            dtrain,
            init_model=model,
            num_boost_round=100,
            valid_sets=[dvalid],
            valid_names=["valid"],
            keep_training_booster=True
        )

    print(f"Trained on batch {i}, file={os.path.basename(path)}")

    score = model.best_score["valid"]["binary_logloss"]
    print(f"batch {i} file={os.path.basename(path)}, valid logloss={score:.5f}")

    if score < best_score - 1e-4:   # small improvement threshold
        best_score = score
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= patience_batches:
            print(f"Stopping: no improvement for {patience_batches} batches.")
            break



Trained on batch 0, file=batch_000400000.parquet
batch 0 file=batch_000400000.parquet, valid logloss=0.11496
Trained on batch 1, file=batch_001400000.parquet
batch 1 file=batch_001400000.parquet, valid logloss=0.11223
Trained on batch 2, file=batch_001200000.parquet
batch 2 file=batch_001200000.parquet, valid logloss=0.10714
Trained on batch 3, file=batch_000200000.parquet
batch 3 file=batch_000200000.parquet, valid logloss=0.10425
Trained on batch 4, file=batch_001000000.parquet
batch 4 file=batch_001000000.parquet, valid logloss=0.10724
Trained on batch 5, file=batch_000000000.parquet
batch 5 file=batch_000000000.parquet, valid logloss=0.10677


In [48]:
from sklearn.metrics import roc_auc_score, log_loss

y_pred_proba = model.predict(X_test)
print("Test logloss:", log_loss(y_test, y_pred_proba))
print("Test AUC:", roc_auc_score(y_test, y_pred_proba))

Test logloss: 0.11710350611191442
Test AUC: 0.842596574197079


In [ ]:







What comes next is garbage!







In [33]:
!ls /content/parquet_files/test/aggregated/

In [ ]:
#
#
#Join aggregated files together into one big parquet (takes too much RAM)
#
#
batch_size = 1
current_lf = lf

for i in range(0, len(agg_paths), batch_size):
  batch = agg_paths[i:i + batch_size]
  print(f"Joining batch {i//batch_size}: {batch}")

   # add joins for this batch
  for p in batch:
      current_lf = current_lf.join(
          pl.scan_parquet(p),
          on="case_id",
          how="left",
      )

  is_last = (i + batch_size) >= len(agg_paths)
  if is_last:
    out_path = AGGR_FILE
  else:
    out_path = os.path.join(
      AGGR_DIR, f"tmp_stage_{i // batch_size}.parquet"
    )

  current_lf.sink_parquet(out_path)
  print(f"Sunk {out_path}")

  if not is_last:
      current_lf = pl.scan_parquet(out_path)

print("Final merged file:", AGGR_FILE)

#lf.sink_parquet(AGGR_FILE)
#lf.head().collect()
#lf.collect_schema().names()


In [ ]:
#
#
# This is how to aggr by names heuristically instead of transform labels
#
#
group = ["other_1.parquet"]
paths = [os.path.join(BASE_DIR, f"train_{p}") for p in group]
print(paths)
tbl = pl.scan_parquet(paths)
schema = tbl.collect_schema()

#get columns by type
dtypes = dict(zip(schema.names(), schema.dtypes()))


P_cols = [c for c, t in dtypes.items() if c.endswith == 'P']
numeric_cols = [c for c, t in dtypes.items() if t in (pl.Int8, pl.Int16, pl.Int32, pl.Int64,
                                                      pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
                                                      pl.Float32, pl.Float64)]
bool_cols    = [c for c, t in dtypes.items() if t == pl.Boolean]
string_cols  = [c for c, t in dtypes.items() if t == pl.Utf8]
date_cols    = [c for c, t in dtypes.items() if t in (pl.Date, pl.Datetime)]

#heuristically identify numeric columns
amount_like = [c for c in numeric_cols if any(
    kw in c.lower() for kw in ["amt", "amount", "value", "payment", "sum"]
)]
other_numeric = [c for c in numeric_cols if c not in amount_like]

#build aggregation list
agg_exprs = [
    pl.len().alias("row_count"),
]

# numeric aggregations (safe defaults)
agg_exprs += [
    pl.col(c).mean().alias(f"{c}_mean") for c in numeric_cols
]
agg_exprs += [
    pl.col(c).std().alias(f"{c}_std") for c in numeric_cols
]
agg_exprs += [
    pl.col(c).min().alias(f"{c}_min") for c in numeric_cols
]
agg_exprs += [
    pl.col(c).max().alias(f"{c}_max") for c in numeric_cols
]

# sum only for amount-like
agg_exprs += [
    pl.col(c).sum().alias(f"{c}_sum") for c in amount_like
]

# boolean
agg_exprs += [
    pl.col(c).mean().alias(f"{c}_share_true") for c in bool_cols
]
agg_exprs += [
    pl.col(c).sum().alias(f"{c}_count_true") for c in bool_cols
]

# categorical: n_unique (you can add more later)
agg_exprs += [
    pl.col(c).n_unique().alias(f"{c}_nunique") for c in string_cols
]

# dates: min/max
agg_exprs += [
    pl.col(c).min().alias(f"{c}_first") for c in date_cols
]
agg_exprs += [
    pl.col(c).max().alias(f"{c}_last") for c in date_cols
]

features = (
    tbl
    .group_by("case_id")
    .agg(agg_exprs)
)

tbl.head().collect()
tbl.fetch(5)